In [1]:
suppressMessages({
    require(Seurat)
    require(dplyr)
    require(igraph)
})

In [2]:
# load orthogroups
orthogroups <- read.delim('/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/02.gene_relationships/run4/results/Ortho_pipeline/OrthoFinder/Orthogroups/Orthogroups.tsv')
# at least one copy for 4 species
orthogroups <- orthogroups %>% select(c('Orthogroup', 'Pmar', 'Pvit', 'Mmus', 'Hsap'))  %>% 
    filter(Pmar != '' | Pvit != '' | Mmus != '' | Hsap != '')

In [3]:
# get TFs for each species
Hsap_TFs <- read.table('/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/01.data/07.species_signals/6.TF_vs_species_signals/TFs/Hsap_TFs.txt', header = F)$V1
Mmus_TFs <- read.table('/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/01.data/07.species_signals/6.TF_vs_species_signals/TFs/Mmus_TFs.name.txt', header = F)$V1
Pvit_TFs <- read.table('/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/01.data/07.species_signals/6.TF_vs_species_signals/TFs/Pvit.predicted_TFs.txt', header = T)
Pmar_TFs <- read.table('/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/01.data/07.species_signals/6.TF_vs_species_signals/TFs/Pmar.predicted_TFs.txt', header = T)
Pvit_TFs <- Pvit_TFs[Pvit_TFs$prediction == 'True', 1]
Pmar_TFs <- Pmar_TFs[Pmar_TFs$prediction == 'True', 1]

In [4]:
# get ohnologues and SSD paralogues info
oh_pa_family <- readRDS('Combined.SSD_WGD.pairs.rds')
paralog_gene_type <- readRDS('gene_type.rds')

In [5]:
# gene annotation for gene ID and gene name
Hsap_ID <- read.delim('0.bin/Hsap.info', header = T)
Hsap_ID <- Hsap_ID[Hsap_ID$Gene.type == 'protein_coding', ]
Hsap_ID[Hsap_ID$Gene.name == '', 'Gene.name'] <- Hsap_ID[Hsap_ID$Gene.name == '', 'Gene.stable.ID']

Mmus_ID <- read.delim('0.bin/Mmus.info', header = T)
Mmus_ID <- Mmus_ID[Mmus_ID$Gene.type == 'protein_coding', ]
Mmus_ID[Mmus_ID$Gene.name == '', 'Gene.name'] <- Mmus_ID[Mmus_ID$Gene.name == '', 'Gene.stable.ID']

In [6]:
Hsap <- readRDS('/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/01.data/02.atlas_final/2.samap/4.final/Hsap.non_neurons.iter_cluster_annotated.rds')
Mmus <- readRDS('/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/01.data/02.atlas_final/2.samap/4.final/Mmus.non_neurons.iter_cluster_annotated.rds')
Pvit <- readRDS('/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/01.data/02.atlas_final/2.samap/4.final/Pvit.non_neurons.iter_cluster_annotated.rds')

In [7]:
# retain  astrocytes and oligodendrocyte lineages
Idents(Hsap) <- 'Refined family'
Idents(Mmus) <- 'Refined family'
Idents(Pvit) <- 'Refined family'
Hsap <- subset(Hsap, idents = c('Astrocytes', 'Oligodendrocyte precursor cells'))
Mmus <- subset(Mmus, idents = c('Astrocytes', 'Oligodendrocyte precursor cells'))
Pvit <- subset(Pvit, idents = c('Astrocytes', 'Oligodendrocyte precursor cells'))

In [8]:
# subset to 5000 maximum
subset_further <- function(obj, number){
    sampled_df <- obj@meta.data
    sampled_df$cellname <- rownames(sampled_df)
    sampled_df <- sampled_df %>% group_by(`Refined family`) %>% slice_sample(n = number) %>% ungroup()
    obj <- subset(obj, cells = as.character(sampled_df$cellname))
    return(obj)
}

Hsap <- subset_further(Hsap, 5000)
Mmus <- subset_further(Mmus, 5000)
Pvit <- subset_further(Pvit, 5000)

In [9]:
# Find markers between AST and OPC
get_markers <- function(object, label){
    markers <- FindAllMarkers(object, only.pos = T, verbose = F)
    markers <- markers %>% filter(p_val_adj < 0.01 & avg_log2FC >= 0.58 & pct.1 > 0.1)
    # rank by pct and avg log2FC
    markers <- markers %>% group_by(cluster) %>% 
        arrange(
            desc(pct.1 / (pct.1 + pct.2)),
            desc(avg_log2FC),
            .by_group = TRUE
        ) %>% ungroup() %>% as.data.frame()
    markers$species <- label
    return(markers)
}

In [10]:
# find markers and add gene names or IDs
Idents(Hsap) <- 'Refined family'
Idents(Mmus) <- 'Refined family'
Idents(Pvit) <- 'Refined family'
Hsap_markers <- get_markers(Hsap, 'Hsap')
Mmus_markers <- get_markers(Mmus, 'Mmus')
Pvit_markers <- get_markers(Pvit, 'Pvit')

In [11]:
# add gene name, TFs, ohnologs vs SSD paralogs
# add gene categories
add_type <- function(markers, label){
    res <- vapply(markers$gene, FUN = function(g){
        tmp <- paralog_gene_type[[label]]
        if (g %in% tmp$gene){
            res <- tmp[tmp$gene == g, 'type']
        } else {
            res <- 'Others'
        }
    }, FUN.VALUE = character(1))
    markers$type <- res
    return(markers)
}

Hsap_markers$gene_name <- Hsap_ID[match(Hsap_markers$gene, Hsap_ID$Gene.stable.ID), 'Gene.name']
Hsap_markers$TF <- Hsap_markers$gene %in% Hsap_TFs
Hsap_markers <- add_type(Hsap_markers, 'Hsap')

Mmus_markers$gene_name <- Mmus_ID[match(Mmus_markers$gene, Mmus_ID$Gene.name), 'Gene.stable.ID']
Mmus_markers$TF <- Mmus_markers$gene %in% Mmus_TFs
Mmus_markers <- add_type(Mmus_markers, 'Mmus')

Pvit_markers$gene_name <- Pvit_markers$gene
Pvit_markers$TF <- Pvit_markers$gene %in% Pvit_TFs
Pvit_markers <- add_type(Pvit_markers, 'Pvit')


In [12]:
# add families information, get family level ohnologs and SSD paralogues
get_family <- function(genes, label){
    # get family in lists
    g <- graph_from_data_frame(oh_pa_family[[label]][,1:2], directed = FALSE)
    components <- components(g)$membership
    family <- split(names(components), components)
    
    # Find the matching family for the gene
    results <- vapply(genes, FUN = function(gene) {
        res <- lapply(family, function(x) {
            if (gene %in% x) return(x)
            })
        # Filter out NULL values (elements where the gene wasn't found)
        res <- Filter(Negate(is.null), res)
        
        # If a match is found, return the first result; otherwise, return an empty string
        if (length(res) > 0) {
            return(paste0(unlist(res), collapse = ","))
        } else {
            return("")
        }
    }, FUN.VALUE = character(1))
    return(results)
}

In [13]:
Hsap_markers$family <- get_family(Hsap_markers$gene, 'Hsap')
Mmus_markers$family <- get_family(Mmus_markers$gene, 'Mmus')
Pvit_markers$family <- get_family(Pvit_markers$gene, 'Pvit')

In [14]:
# add orthogroup information
get_orthogroup <- function(markers, species){
    tmp = orthogroups[, c('Orthogroup', species)] %>% filter(species != '') %>% 
        tidyr::separate_rows(species, sep = ", ") %>% as.data.frame()
    tmp <- tmp[match(markers, tmp[[species]]),1]
    return(tmp)
}

Hsap_markers$orthogroup <- get_orthogroup(Hsap_markers$gene, 'Hsap')
Mmus_markers$orthogroup <- get_orthogroup(Mmus_markers$gene, 'Mmus')
Pvit_markers$orthogroup <- get_orthogroup(Pvit_markers$gene, 'Pvit')

Warning message:
“Using an external vector in selections was deprecated in tidyselect 1.1.0.
ℹ Please use `all_of()` or `any_of()` instead.
  # Was:
  data %>% select(species)

  # Now:
  data %>% select(all_of(species))

See <https://tidyselect.r-lib.org/reference/faq-external-vector.html>.”


In [15]:
# function to get WGD paralogue family with different members used in AST and Oligo OR 
# WGD paralogue family only used in one of above two cell types.

get_markers_for_divergence_WGD <- function(markers, label){
    markers <- markers %>% filter(type == 'WGD')
    tmp1 <- unique(unlist(markers %>% filter(cluster == 'Astrocytes') %>% select(family)))
    tmp2 <- unique(unlist(markers %>% filter(cluster == 'Oligodendrocyte precursor cells') %>% select(family)))
    tmp1 <- tmp1[tmp1 != '']
    tmp2 <- tmp2[tmp2 != '']
    
    cat(paste0(label, ':\nNumber of WGD paralogue family involved:', length(unique(c(tmp1,tmp2))),
               ';\nNumber of WGD paralogue family involved only in one of AST and OPC:', sum(table(c(tmp1,tmp2)) == 1),
               ';\nNumber of WGD paralogue family involved in these two:', sum(table(c(tmp1,tmp2)) == 2), '\n'))
    
    x <- (table(c(tmp1,tmp2)) == 2)
    interested_1 <- names(x)[x]
    markers_1 <- markers %>% filter(family %in% interested_1)
    
    x <- (table(c(tmp1,tmp2)) == 1)
    interested_2 <- names(x)[x]
    markers_2 <- markers %>% filter(family %in% interested_2)
    return(list(a = markers_1, b = markers_2))
}

# function to get SSD paralogue family with different members used in AST and Oligo
get_markers_for_divergence_SSD <- function(markers, label){
    markers <- markers %>% filter(type == 'SSD')
    tmp1 <- unique(unlist(markers %>% filter(cluster == 'Astrocytes') %>% select(family)))
    tmp2 <- unique(unlist(markers %>% filter(cluster == 'Oligodendrocyte precursor cells') %>% select(family)))
    tmp1 <- tmp1[tmp1 != '']
    tmp2 <- tmp2[tmp2 != '']
    
    cat(paste0(label, ':\nNumber of SSD paralogue family involved:', length(unique(c(tmp1,tmp2))),
               ';\nNumber of SSD paralogue family involved only in one of AST and OPC:', sum(table(c(tmp1,tmp2)) == 1),
               ';\nNumber of SSD paralogue family involved in these two:', sum(table(c(tmp1,tmp2)) == 2), '\n'))
    
    x <- (table(c(tmp1,tmp2)) == 2)
    interested_1 <- names(x)[x]
    markers_1 <- markers %>% filter(family %in% interested_1)
    
    x <- (table(c(tmp1,tmp2)) == 1)
    interested_2 <- names(x)[x]
    markers_2 <- markers %>% filter(family %in% interested_2)
    return(list(a = markers_1, b = markers_2))
}

In [16]:
Hsap_interesting_WGD <- get_markers_for_divergence_WGD(Hsap_markers, 'Hsap')
Hsap_interesting_SSD <- get_markers_for_divergence_SSD(Hsap_markers, 'Hsap')
Mmus_interesting_WGD <- get_markers_for_divergence_WGD(Mmus_markers, 'Mmus')
Mmus_interesting_SSD <- get_markers_for_divergence_SSD(Mmus_markers, 'Mmus')
Pvit_interesting_WGD <- get_markers_for_divergence_WGD(Pvit_markers, 'Pvit')
Pvit_interesting_SSD <- get_markers_for_divergence_SSD(Pvit_markers, 'Pvit')

Hsap:
Number of WGD paralogue family involved:654;
Number of WGD paralogue family involved only in one of AST and OPC:584;
Number of WGD paralogue family involved in these two:70
Hsap:
Number of SSD paralogue family involved:474;
Number of SSD paralogue family involved only in one of AST and OPC:440;
Number of SSD paralogue family involved in these two:34
Mmus:
Number of WGD paralogue family involved:389;
Number of WGD paralogue family involved only in one of AST and OPC:376;
Number of WGD paralogue family involved in these two:13
Mmus:
Number of SSD paralogue family involved:322;
Number of SSD paralogue family involved only in one of AST and OPC:312;
Number of SSD paralogue family involved in these two:10
Pvit:
Number of WGD paralogue family involved:390;
Number of WGD paralogue family involved only in one of AST and OPC:369;
Number of WGD paralogue family involved in these two:21
Pvit:
Number of SSD paralogue family involved:214;
Number of SSD paralogue family involved only in one of

In [17]:
# AST conserved marker family, Oligo conserved marker family
tmp1 = c(
    unique(unlist(Hsap_markers %>% filter(cluster == 'Astrocytes' & TF) %>% select('orthogroup'))),
    unique(unlist(Mmus_markers %>% filter(cluster == 'Astrocytes' & TF) %>% select('orthogroup'))),
    unique(unlist(Pvit_markers %>% filter(cluster == 'Astrocytes' & TF) %>% select('orthogroup')))
)

tmp2 = c(
    unique(unlist(Hsap_markers %>% filter(cluster == 'Oligodendrocyte precursor cells' & TF) %>% select('orthogroup'))),
    unique(unlist(Mmus_markers %>% filter(cluster == 'Oligodendrocyte precursor cells' & TF) %>% select('orthogroup'))),
    unique(unlist(Pvit_markers %>% filter(cluster == 'Oligodendrocyte precursor cells' & TF) %>% select('orthogroup')))
)

tmp1 = names(table(tmp1))[table(tmp1) == 3]
tmp2 = names(table(tmp2))[table(tmp2) == 3]

Amniote_AST_TF <- rbind(Hsap_markers, Mmus_markers, Pvit_markers) %>% 
        filter(orthogroup %in% tmp1 & TF & cluster == 'Astrocytes')
Amniote_OPC_TF <- rbind(Hsap_markers, Mmus_markers, Pvit_markers) %>% 
        filter(orthogroup %in% tmp2 & TF & cluster == 'Oligodendrocyte precursor cells')



In [19]:
length(unique(c(unique(Amniote_AST_TF$orthogroup), unique(Amniote_OPC_TF$orthogroup))))

[1] 8

In [21]:
write.table(rbind(Amniote_AST_TF, Amniote_OPC_TF), file = 'Amniote_conserved.AST_vs_OPC.TF_orthogroup.txt', 
            quote = F, sep = '\t', row.names= F, col.names = T)

In [24]:
Amniote_AST_Oligo_TF <- read.delim('Amniote_conserved.AST_vs_Oligo.TF_orthogroup.txt', header = T)

In [29]:
Amniote_OPC_TF %>% filter(species == 'Hsap') %>% arrange(orthogroup)
Amniote_OPC_TF %>% filter(species == 'Mmus') %>% arrange(orthogroup)
Amniote_OPC_TF %>% filter(species == 'Pvit') %>% arrange(orthogroup)

p_val,avg_log2FC,pct.1,pct.2,p_val_adj,cluster,gene,species,gene_name,TF,type,family,orthogroup
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<chr>,<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>
1.022964e-168,6.4312983,0.148,0.002,2.013193e-164,Oligodendrocyte precursor cells,ENSG00000100146,Hsap,SOX10,TRUE,WGD,"ENSG00000100146,ENSG00000005513,ENSG00000125398",OG0000843
3.951856e-242,3.6709860,0.247,0.020,7.777253e-238,Oligodendrocyte precursor cells,ENSG00000005513,Hsap,SOX8,TRUE,WGD,"ENSG00000100146,ENSG00000005513,ENSG00000125398",OG0000843
1.444190e-162,2.2835283,0.255,0.058,2.842166e-158,Oligodendrocyte precursor cells,ENSG00000244405,Hsap,ETV5,TRUE,SSD,"ENSG00000006468,ENSG00000175832,ENSG00000244405",OG0000883
8.736636e-167,1.6313224,0.381,0.144,1.719370e-162,Oligodendrocyte precursor cells,ENSG00000006468,Hsap,ETV1,TRUE,SSD,"ENSG00000006468,ENSG00000175832,ENSG00000244405",OG0000883
4.118494e-21,0.7384193,0.114,0.060,8.105196e-17,Oligodendrocyte precursor cells,ENSG00000143842,Hsap,SOX13,TRUE,WGD,"ENSG00000134532,ENSG00000143842,ENSG00000110693",OG0001046
0.000000e+00,2.3357776,0.989,0.748,0.000000e+00,Oligodendrocyte precursor cells,ENSG00000110693,Hsap,SOX6,TRUE,WGD,"ENSG00000134532,ENSG00000143842,ENSG00000110693",OG0001046
0.000000e+00,5.8158348,0.374,0.008,0.000000e+00,Oligodendrocyte precursor cells,ENSG00000205927,Hsap,OLIG2,TRUE,WGD,"ENSG00000177468,ENSG00000205927",OG0001989
0.000000e+00,6.4386896,0.779,0.022,0.000000e+00,Oligodendrocyte precursor cells,ENSG00000184221,Hsap,OLIG1,TRUE,Others,,OG0013365


p_val,avg_log2FC,pct.1,pct.2,p_val_adj,cluster,gene,species,gene_name,TF,type,family,orthogroup
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<chr>,<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>
0.000000e+00,5.6735898,0.343,0.004,0.000000e+00,Oligodendrocyte precursor cells,Sox10,Mmus,ENSMUSG00000033006,TRUE,WGD,"Sox10,Sox8,Sox9",OG0000843
2.870984e-21,0.6723745,0.302,0.149,5.393717e-17,Oligodendrocyte precursor cells,Sox8,Mmus,ENSMUSG00000024176,TRUE,WGD,"Sox10,Sox8,Sox9",OG0000843
4.975252e-14,0.9240277,0.101,0.037,9.347006e-10,Oligodendrocyte precursor cells,Etv1,Mmus,ENSMUSG00000004151,TRUE,SSD,"Etv1,Etv4,Etv5",OG0000883
7.625721e-14,0.6703274,0.183,0.088,1.432644e-09,Oligodendrocyte precursor cells,Etv4,Mmus,ENSMUSG00000017724,TRUE,SSD,"Etv1,Etv4,Etv5",OG0000883
1.554326e-38,1.1176759,0.266,0.093,2.920112e-34,Oligodendrocyte precursor cells,Sox6,Mmus,ENSMUSG00000051910,TRUE,WGD,"Sox6,Sox13,Sox5",OG0001046
0.000000e+00,3.2394820,0.724,0.094,0.000000e+00,Oligodendrocyte precursor cells,Olig2,Mmus,ENSMUSG00000039830,TRUE,WGD,"Olig2,Olig3",OG0001989
0.000000e+00,3.9806680,0.986,0.223,0.000000e+00,Oligodendrocyte precursor cells,Olig1,Mmus,ENSMUSG00000046160,TRUE,Others,,OG0013365


p_val,avg_log2FC,pct.1,pct.2,p_val_adj,cluster,gene,species,gene_name,TF,type,family,orthogroup
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<chr>,<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>
5.865359e-289,4.953198,0.256,0.009,1.108025e-284,Oligodendrocyte precursor cells,SOX10,Pvit,SOX10,TRUE,WGD,"SOX9,SOX10,SOX8",OG0000843
0.000000e+00,4.457167,0.792,0.064,0.000000e+00,Oligodendrocyte precursor cells,SOX8,Pvit,SOX8,TRUE,WGD,"SOX9,SOX10,SOX8",OG0000843
6.059991e-41,1.510160,0.105,0.036,1.144793e-36,Oligodendrocyte precursor cells,ETV5,Pvit,ETV5,TRUE,WGD,"ETV5,ETV1",OG0000883
1.091872e-59,1.587907,0.153,0.054,2.062656e-55,Oligodendrocyte precursor cells,ETV1,Pvit,ETV1,TRUE,WGD,"ETV5,ETV1",OG0000883
8.867094e-194,3.132516,0.225,0.028,1.675083e-189,Oligodendrocyte precursor cells,SOX6,Pvit,SOX6,TRUE,WGD,"SOX13,SOX5,SOX6",OG0001046
2.716210e-72,1.310943,0.240,0.105,5.131192e-68,Oligodendrocyte precursor cells,SOX5,Pvit,SOX5,TRUE,WGD,"SOX13,SOX5,SOX6",OG0001046
0.000000e+00,5.223650,0.641,0.030,0.000000e+00,Oligodendrocyte precursor cells,OLIG2,Pvit,OLIG2,TRUE,WGD,"OLIG3,OLIG2",OG0001989
0.000000e+00,4.651171,0.823,0.086,0.000000e+00,Oligodendrocyte precursor cells,OLIG1,Pvit,OLIG1,TRUE,Others,,OG0013365


In [30]:
Amniote_AST_TF %>% filter(species == 'Hsap') %>% arrange(orthogroup)
Amniote_AST_TF %>% filter(species == 'Mmus') %>% arrange(orthogroup)
Amniote_AST_TF %>% filter(species == 'Pvit') %>% arrange(orthogroup)

p_val,avg_log2FC,pct.1,pct.2,p_val_adj,cluster,gene,species,gene_name,TF,type,family,orthogroup
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<chr>,<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>
7.077069e-78,2.0998590,0.170,0.054,1.392767e-73,Astrocytes,ENSG00000184486,Hsap,POU3F2,TRUE,WGD,"ENSG00000184486,ENSG00000198914,ENSG00000185668",OG0000114
2.238328e-67,0.8265529,0.496,0.353,4.405029e-63,Astrocytes,ENSG00000143190,Hsap,POU2F1,TRUE,SSD,"ENSG00000028277,ENSG00000064835,ENSG00000137709,ENSG00000143190,ENSG00000196767,ENSG00000204531,ENSG00000212993",OG0000114
7.096920e-247,3.0914520,0.298,0.047,1.396674e-242,Astrocytes,ENSG00000175745,Hsap,NR2F1,TRUE,WGD,"ENSG00000175745,ENSG00000160113,ENSG00000185551",OG0000810
0.000000e+00,8.1837231,0.348,0.002,0.000000e+00,Astrocytes,ENSG00000172201,Hsap,ID4,TRUE,Others,,OG0000815
0.000000e+00,6.1356234,0.331,0.014,0.000000e+00,Astrocytes,ENSG00000115738,Hsap,ID2,TRUE,WGD,"ENSG00000117318,ENSG00000125968,ENSG00000115738",OG0000815
1.234125e-296,7.1005235,0.247,0.003,2.428758e-292,Astrocytes,ENSG00000125398,Hsap,SOX9,TRUE,WGD,"ENSG00000100146,ENSG00000005513,ENSG00000125398",OG0000843


p_val,avg_log2FC,pct.1,pct.2,p_val_adj,cluster,gene,species,gene_name,TF,type,family,orthogroup
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<chr>,<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>
1.775021e-12,1.456656,0.214,0.112,3.334733e-08,Astrocytes,Pou3f3,Mmus,ENSMUSG00000045515,TRUE,WGD,"Pou3f1,Pou3f2,Pou3f3",OG0000114
2.471210e-15,2.217794,0.192,0.079,4.642662e-11,Astrocytes,Nr2f1,Mmus,ENSMUSG00000069171,TRUE,WGD,"Nr2f2,Nr2f1,Nr2f6",OG0000810
5.519385e-51,6.535099,0.259,0.007,1.036927e-46,Astrocytes,Id4,Mmus,ENSMUSG00000021379,TRUE,Others,,OG0000815
4.586261e-20,3.885337,0.134,0.017,8.616208e-16,Astrocytes,Id1,Mmus,ENSMUSG00000042745,TRUE,WGD,"Id1,Id3,Id2",OG0000815
1.620293e-34,1.771477,0.471,0.269,3.044044e-30,Astrocytes,Id2,Mmus,ENSMUSG00000020644,TRUE,WGD,"Id1,Id3,Id2",OG0000815
3.981294e-97,6.583648,0.424,0.010,7.479657e-93,Astrocytes,Sox9,Mmus,ENSMUSG00000000567,TRUE,WGD,"Sox10,Sox8,Sox9",OG0000843


p_val,avg_log2FC,pct.1,pct.2,p_val_adj,cluster,gene,species,gene_name,TF,type,family,orthogroup
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<chr>,<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>
3.640296e-38,1.336464,0.120,0.048,6.876884e-34,Astrocytes,POU3F3,Pvit,POU3F3,TRUE,WGD,"LOC110070984,POU3F3",OG0000114
1.890635e-155,1.992096,0.320,0.108,3.571599e-151,Astrocytes,NR2F2,Pvit,NR2F2,TRUE,WGD,"NR2F2,NR2F6",OG0000810
0.000000e+00,5.315626,0.858,0.073,0.000000e+00,Astrocytes,ID4,Pvit,ID4,TRUE,WGD,"ID4,ID3",OG0000815
0.000000e+00,5.805556,0.490,0.012,0.000000e+00,Astrocytes,SOX9,Pvit,SOX9,TRUE,WGD,"SOX9,SOX10,SOX8",OG0000843
